### 模型二：

In [93]:
import pandas as pd
from pulp import LpMinimize, LpProblem, LpVariable, lpSum, LpBinary,LpStatus
import collections
from itertools import product



file_path = "realdata_v5_2_new.xlsx"  
activities_df = pd.read_excel(file_path, sheet_name="ActivitiesInfo")
classroom_df = pd.read_excel(file_path, sheet_name="ClassroomsInfo")
student_courses_df = pd.read_excel(file_path, sheet_name="StudentsInfo")



In [94]:
df = pd.read_csv("processed_schedule.csv")
# df = df[(df["Week"] == 3) & (df["Time_slot"] == 3)]
df


,Week,Time_slot,subject_candidates
0,1,1,"{43, 11}"
1,1,2,{35}
2,1,3,"{5, 40, 13, 16, 48, 18, 52}"
3,1,4,"{35, 28}"
4,1,5,{33}
...,...,...,...
79,11,7,{7}
80,12,1,{2}
81,12,3,{33}
82,12,7,{6}


In [95]:
courses = student_courses_df["Course_ID"]
courses = set([int(num) for row in courses for num in row.split(', ')])

classrooms = classroom_df["Classroom_ID"]
classrooms = set(int(row) for row in classrooms)

activity_types = activities_df["Activity_Type"]
activity_types = set(activity_types)

In [96]:
# parameters
Pc = [row["Classroom_ID"] for _, row in classroom_df.iterrows() if row["Has_Computers"] == 1]
Ysi = {(row["Course_ID"], row["Activity_Type"]): 1 for _, row in activities_df.iterrows()}
Gs = {row["Course_ID"]: 1 for _, row in activities_df.iterrows() if row["Requires_Separation"] == 1}
Cs = {row["Course_ID"]: 1 for _, row in activities_df.iterrows() if row["Requires_Computers"] == 1}
Ts = {row["Course_ID"]: 1 for _, row in activities_df.iterrows() if row["Requires_Tables"] == 1}
Tc = {row["Classroom_ID"]: 1 for _, row in classroom_df.iterrows() if row["Has_Tables"] == 1}
Act = {(row["Classroom_ID"], int(w), int(t)): 1 for _, row in classroom_df.iterrows() for w in row["Available_Weeks"].split(', ') for t in row["Time_Slot"].split(', ')} #
Ic = {row["Classroom_ID"]: 1 for _, row in classroom_df.iterrows() if row["Is_Isolated"] == 1}
Ns = {row["Course_ID"]: row["Num_Students"] for _, row in activities_df.iterrows()}
Msg = collections.defaultdict(int)

for _, row in student_courses_df.iterrows():
    class_id = row["Class_ID"]
    course_ids = list(map(int, str(row["Course_ID"]).split(', ')))  
    for course_id in course_ids:
        Msg[(course_id, class_id)] += 1  


class_courses = collections.defaultdict(set)
for (course, class_id), _ in Msg.items():
    class_courses[course].add(class_id)

Capacity = {row["Classroom_ID"]: row["Capacity"] for _, row in classroom_df.iterrows()}
Occupancy_rate = {1: 1, 2: 0.9, 3: 0}
CAPci = {(c, i): int(Capacity[c] * Occupancy_rate[i]) for c, i in product(classrooms, activity_types)}

In [99]:
import csv
with open("results-容量提升.csv", "w", newline="", encoding="utf-8") as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(["Week", "Time_slot", "subject_candidates", "result","value"])

results = []

In [100]:
for count in range(len(df)):
    week = df.iloc[count]["Week"]
    time_slot = df.iloc[count]["Time_slot"]
    subjects = df["subject_candidates"].tolist()
    subjects = [int(i.strip('{}')) for i in subjects[count].split(', ')]
    # 2. Define the model
    model = LpProblem(name="classroom_assignment", sense=LpMinimize)

    z = {(c, s): LpVariable(f"z_{c}_{s}", cat=LpBinary) for c in classrooms for s in subjects}
    w = {(c, s, g): LpVariable(f"w_{c}_{s}_{g}", cat=LpBinary) for c in classrooms for s in subjects for g in class_courses[s]}
   
    # Add slack variables for capacity relaxation
    eta = {(s): LpVariable(f"eta_{s}", lowBound=0) for s in subjects}
    lamda = 20

    # Objective function
    model += lpSum(z[c, s] + lamda * eta[s] for c in classrooms for s in subjects), "Minimize_Classroom_Usage"

    # A.13 - Ensure classroom capacity is sufficient
    # for s in subjects:
    #     model += lpSum(z[c, s] * sum(CAPci.get((c, i), 0) * Ysi.get((s, i), 0) for i in activity_types) for c in classrooms) >= Ns[s], f"Capacity_Constraint_{s}"
    ## Modification: Add slack constraints
    for s in subjects:
        model += lpSum(z[c, s] * sum(CAPci.get((c, i), 0) * Ysi.get((s, i), 0) for i in activity_types) for c in classrooms) + eta[s] >= Ns[s], f"Capacity_Constraint_{s}"
        model += eta[s] <= 0.15 * Ns[s]

    # A.14 - Ensure classroom capacity is sufficient for each class
    for s in subjects:
        if Ts.get(s, 0) == 1:
            for g in class_courses[s]:
                model += lpSum(w[c, s, g] * sum(CAPci.get((c, i), 0) * Ysi.get((s, i), 0) for i in activity_types) for c in classrooms) >= Msg.get((s, g), 0), f"Classroom_Capacity_{s}_{g}"

    # A.15 - Prevent classrooms from being reused
    for c in classrooms:
        model += lpSum(z[c, s] for s in subjects) <= 1, f"Single_Assignment_{c}"

    # A.16 - Only allow available classrooms to be assigned
    for c in classrooms:
        for s in subjects:
            model += z[c, s] <= Act.get((c, week, time_slot), 0), f"Classroom_Availability_{c}_{s}"

    # A.17 - Consistency in class hour allocation
    for c in classrooms:
        for s in subjects:
            if Gs.get(s, 0) == 1:
                for g in class_courses[s]:
                    model += z[c, s] >= w[c, s, g], f"Consistency_1_{c}_{s}_{g}"

    # A.18 - Consistency in class hour allocation
    for c in classrooms:
        for s in subjects:
            if Gs.get(s, 0) == 1:
                model += z[c, s] <= lpSum(w[c, s, g] for g in class_courses[s]), f"Consistency_2_{c}_{s}"

    # A.19 - Ensure different classes do not share classrooms
    for c in classrooms:
        for s in subjects:
            if Gs.get(s, 0) == 1:
                for g1 in class_courses[s]:
                    for g2 in class_courses[s]:
                        if g1 != g2:
                            model += w[c, s, g1] + w[c, s, g2] <= 1, f"No_Shared_Classroom_{c}_{s}_{g1}_{g2}"

    # A.20 - Computer room requirements
    for c in classrooms:
        for s in subjects:
            if Cs.get(s, 0) == 1 and c not in Pc:
                model += z[c, s] == 0, f"Computer_Requirement_{c}_{s}"

    # A.21 - Desk requirements
    for c in classrooms:
        for s in subjects:
            if Ts.get(s, 0) == 1 and Tc.get(c, 0) == 0:  # Course requires desks but classroom does not have them
                model += z[c, s] == 0, f"Table_Requirement_{c}_{s}"

    # A.22 - Isolation classrooms
    for c in classrooms:
        if Ic.get(c, 0) == 1:
            for s in subjects:
                model += lpSum(z[c_prime, s] for c_prime in classrooms if c_prime != c) <= (1 - z[c, s]) * len(classrooms), f"Isolation_Requirement_{c}_{s}"

    # 3. Solve the model
    model.solve()

    # Output optimization results
    print("Week:", week)
    print("Time Slot:", time_slot)
    print("Optimization Status:", LpStatus[model.status])
    value = model.objective.value()
    print("Objective Function Value:", value)
    if model.status == 1:  # Output assignment details only if a feasible solution is found
        print("\nClassroom Assignment Details:")
        assigned_classrooms = []
        for (c, s), var in z.items():
            if var.value() == 1:
                assigned_classrooms.append((c, s))
                print(f" - Classroom {c} assigned to Course {s}")
    else:
        print("\n!! No feasible solution found, please check the constraints !!")
        # Output conflicting constraints (if any)
        for name, constraint in model.constraints.items():
            if not constraint.valid():
                print(f"Conflicting Constraint: {name}")

    # Collect and write results within the loop
    if model.status == 1:
        # Collect results for the current iteration
        value = model.objective.value()
        assignment_pairs = []
        for (c, s), var in z.items():
            if var.value() == 1:
                # Sort by course ascending, classroom ascending
                assignment_pairs.append((s, c))
                print(assignment_pairs)
        
        # Generate the result string as required
        sorted_assignments = sorted(assignment_pairs, key=lambda x: (int(x[0]), int(x[1])))
        result_entries = [f"{{{s}:{c}}}" for s, c in sorted_assignments]
        result_str = ",".join(result_entries)

        # Generate subject_candidates (original logic retained)
        subject_set = {s for s, _ in sorted_assignments}
        sorted_subjects = sorted(subject_set, key=int)
        subject_candidates = "{%s}" % ",".join(map(str, sorted_subjects))
    else:
        subject_candidates = "{}"
        result_str = "{}"

    # Write to CSV (retain original writing logic)
    with open("results-capacity-enhancement.csv", "a", newline="", encoding="utf-8") as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow([
            df.iloc[count]["Week"],
            df.iloc[count]["Time_slot"],
            df.iloc[count]["subject_candidates"],
            result_str,
            value
        ])


Welcome to the CBC MILP Solver 
Version: 2.10.3 
Build Date: Dec 15 2019 

command line - /opt/anaconda3/lib/python3.12/site-packages/pulp/solverdir/cbc/osx/arm64/cbc /var/folders/xg/6886p3rj5kb44k2smttjctf80000gn/T/c2becbb96c0741178fbaa65b8c2d9b0d-pulp.mps -timeMode elapsed -branch -printingOptions all -solution /var/folders/xg/6886p3rj5kb44k2smttjctf80000gn/T/c2becbb96c0741178fbaa65b8c2d9b0d-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 533 COLUMNS
At line 2935 RHS
At line 3464 BOUNDS
At line 3807 ENDATA
Problem MODEL has 528 rows, 344 columns and 1601 elements
Coin0008I MODEL read with 0 errors
Option for timeMode changed from cpu to elapsed
Continuous objective value is 0.801528 - 0.00 seconds
Cgl0002I 90 variables fixed
Cgl0003I 0 fixed, 0 tightened bounds, 18 strengthened rows, 0 substitutions
Cgl0003I 0 fixed, 0 tightened bounds, 9 strengthened rows, 0 substitutions
Cgl0004I processed model has 31 rows, 39 columns (37 integer (36 of which bin

In [91]:
Msg[6,24]


33

In [17]:
# 统计结果
df_result = pd.read_csv("results-容量提升.csv")

df_result

#统计 subject_candidates 的{}数量

subject_candidates = df_result["subject_candidates"].tolist()
subject_candidates = pd.Series(subject_candidates)
subject_candidates.value_counts()


{11}               8
{35}               4
{45}               4
{33}               4
{48}               4
{30}               4
{43}               3
{2}                3
{1}                3
{32}               2
{47}               2
{38}               2
{40}               2
{53}               2
{15}               2
{1, 3, 28}         1
{16, 6}            1
{52, 13}           1
{5, 47}            1
{32, 18}           1
{10, 52}           1
{1, 18, 19, 42}    1
{14}               1
{18, 5, 15}        1
{21}               1
{48, 28, 44}       1
{50, 7}            1
{12}               1
{20}               1
{18, 45}           1
{4}                1
{41}               1
{52, 37}           1
{7}                1
{10}               1
{27}               1
{33, 20}           1
{10, 11}           1
{33, 3, 6}         1
{16, 52, 13}       1
{53, 13}           1
{28, 21}           1
{1, 10, 19}        1
{48, 7}            1
{42, 35, 13}       1
{51}               1
{40, 13}           1
{10, 52, 13} 

In [18]:
Gs.get(s, 0)                    

1

In [19]:
print("课程15的活动类型及对应容量:")
for s in subjects:
    for g in class_courses[s]:
        print("课程：",s,"班级：",g)
        for c in classrooms:
            print ("左class",c, " zhi:",sum(CAPci.get((c, i), 0) * Ysi.get((s, i), 0) for i in activity_types))
            print("教室人数 ：", Msg.get((s,g),0))

print("课程15的活动类型及对应容量:")

课程15的活动类型及对应容量:
课程： 30 班级： 12
左class 1  zhi: 163
教室人数 ： 80
左class 2  zhi: 66
教室人数 ： 80
左class 3  zhi: 62
教室人数 ： 80
左class 4  zhi: 594
教室人数 ： 80
左class 5  zhi: 167
教室人数 ： 80
左class 6  zhi: 311
教室人数 ： 80
左class 7  zhi: 380
教室人数 ： 80
左class 8  zhi: 256
教室人数 ： 80
左class 9  zhi: 76
教室人数 ： 80
左class 10  zhi: 102
教室人数 ： 80
左class 11  zhi: 370
教室人数 ： 80
左class 12  zhi: 47
教室人数 ： 80
左class 13  zhi: 178
教室人数 ： 80
左class 14  zhi: 437
教室人数 ： 80
左class 15  zhi: 68
教室人数 ： 80
左class 16  zhi: 136
教室人数 ： 80
左class 17  zhi: 117
教室人数 ： 80
左class 18  zhi: 26
教室人数 ： 80
左class 19  zhi: 45
教室人数 ： 80
左class 20  zhi: 45
教室人数 ： 80
左class 21  zhi: 159
教室人数 ： 80
左class 22  zhi: 57
教室人数 ： 80
左class 23  zhi: 74
教室人数 ： 80
左class 25  zhi: 91
教室人数 ： 80
左class 26  zhi: 79
教室人数 ： 80
左class 27  zhi: 95
教室人数 ： 80
左class 28  zhi: 95
教室人数 ： 80
左class 29  zhi: 72
教室人数 ： 80
左class 30  zhi: 98
教室人数 ： 80
左class 31  zhi: 114
教室人数 ： 80
左class 32  zhi: 374
教室人数 ： 80
左class 33  zhi: 285
教室人数 ： 80
左class 34  zhi: 197
教室人数 ： 80
左clas

In [20]:
# 遍历所有约束，筛选涉及教室A14的约束
for name, constraint in model.constraints.items():
    if "Capacity_Constraint_" in name or any("Capacity_Constraint_" in str(var) for var in constraint.values()):
        print(f"约束名称: {name}")
        print(f"约束表达式: {constraint}")
        print(f"约束是否有效: {constraint.valid()}\n")



for name, constraint in model.constraints.items():
    if "Classroom_Capacity_" in name or any("Classroom_Capacity_" in str(var) for var in constraint.values()):
        print(f"约束名称: {name}")
        print(f"约束表达式: {constraint}")
        print(f"约束是否有效: {constraint.valid()}\n")

约束名称: Capacity_Constraint_30
约束表达式: eta_30 + 102*z_10_30 + 370*z_11_30 + 47*z_12_30 + 178*z_13_30 + 437*z_14_30 + 68*z_15_30 + 136*z_16_30 + 117*z_17_30 + 26*z_18_30 + 45*z_19_30 + 163*z_1_30 + 45*z_20_30 + 159*z_21_30 + 57*z_22_30 + 74*z_23_30 + 91*z_25_30 + 79*z_26_30 + 95*z_27_30 + 95*z_28_30 + 72*z_29_30 + 66*z_2_30 + 98*z_30_30 + 114*z_31_30 + 374*z_32_30 + 285*z_33_30 + 197*z_34_30 + 129*z_35_30 + 38*z_36_30 + 45*z_37_30 + 171*z_38_30 + 114*z_39_30 + 62*z_3_30 + 38*z_40_30 + 38*z_41_30 + 36*z_42_30 + 38*z_43_30 + 87*z_44_30 + 53*z_45_30 + 38*z_46_30 + 45*z_47_30 + 199*z_48_30 + 760*z_49_30 + 594*z_4_30 + 570*z_50_30 + 250*z_51_30 + 114*z_52_30 + 285*z_53_30 + 570*z_54_30 + 212*z_55_30 + 95*z_56_30 + 110*z_57_30 + 710*z_58_30 + 167*z_5_30 + 311*z_6_30 + 380*z_7_30 + 256*z_8_30 + 76*z_9_30 >= 208.0
约束是否有效: True

约束名称: Classroom_Capacity_30_12
约束表达式: 102*w_10_30_12 + 370*w_11_30_12 + 47*w_12_30_12 + 178*w_13_30_12 + 437*w_14_30_12 + 68*w_15_30_12 + 136*w_16_30_12 + 117*w_17_30_12 + 